# Suite PRX — Proxies

The dynamic implementation: registry, read-only instances, the builder DSL, create/clone/update, relation entries,
schema inference through links, entry removal, and the behaviors Framework.md pins down.

In [ ]:
import { Plain, Proxies, Schemas as S, Validators } from "../src/Framework/index.js";
import { AttributeError, LookupError, ValueError } from "../src/Framework/Errors.js";
import { assert, entries, get_property, raises, same, set_property, text } from "./support.js";

let Person = new S.OfObject.Builder().properties(text("name"), text("age", BigInt), text("score", Number), text("ok", Boolean),
  text("blob", Uint8Array)).create();
let Pet = new S.OfObject.Builder().properties(text("name")).create();
let Robot = new S.OfObject.Builder().properties(text("name")).create();
const Owns = new S.OfRelation.Builder().links("owner", "item").properties(text("since"), text("share", Number),
  text("rank", BigInt), text("flag", Boolean)).create();
const Knows = new S.OfRelation.Builder().links("a", "b").create();
const Trio = new S.OfRelation.Builder().links("x", "y", "z").create();
Person = new S.OfObject.Builder(Person).relations(
  (r) => r.name("pets").of(Owns).me("owner"),
  (r) => r.name("knows").of(Knows).me("a"),
  (r) => r.name("known_by").of(Knows).me("b"),
  (r) => r.name("trio").of(Trio).me("x"),
).update();
Pet = new S.OfObject.Builder(Pet).relations((r) => r.name("owners").of(Owns).me("item"), (r) => r.name("trio_y").of(Trio).me("y")).update();
Robot = new S.OfObject.Builder(Robot).relations((r) => r.name("owners").of(Owns).me("item"), (r) => r.name("trio_z").of(Trio).me("z")).update();
for (const [name, schema] of [["Person", Person], ["Pet", Pet], ["Robot", Robot], ["Owns", Owns], ["Knows", Knows], ["Trio", Trio]] as const) {
  Proxies.register(name, schema);
}
const B = Proxies.Builders;

## PRX-01 · Registry

In [ ]:
raises(ValueError, () => Proxies.register("Person", Person), "already registered");
raises(AttributeError, () => B.Nobody(), "no schema registered");
raises(TypeError, () => B.Owns(), "no relation builder");
assert(Proxies.schema("Person") === Person && B.schema("Person") === Person);
assert(B.then === undefined && B[Symbol.toStringTag] === undefined); // not a registered name
assert(Proxies.name_of(Owns) === "Owns" && B.name_of(Person) === "Person");
raises(LookupError, () => Proxies.name_of(new S.OfObject.Data()));
raises(TypeError, () => Proxies.schema("Owns")); // a relation is not an object schema

Proxies.register("iso3166.Country", new S.OfObject.Builder().properties(text("alpha2")).create());
const fr = B["iso3166.Country"]().alpha2("FR").create();
assert(fr.schema_name() === "iso3166.Country" && fr.alpha2 === "FR");

## PRX-02 · Instances: read-only attributes; unset properties throw `AttributeError`; identities never repeat

In [ ]:
const p = B.Person().name("Ada").create();
assert(p.name === "Ada");
raises(AttributeError, () => p.age, "'age' is not set");
raises(AttributeError, () => p.no_such_property);
assert(!("age" in p) && "name" in p);
const getattr = (obj: any, name: string, fallback: unknown) => { try { return obj[name]; } catch (e) { if (e instanceof AttributeError) return fallback; throw e; } };
assert(getattr(p, "age", "default") === "default");
raises(AttributeError, () => { (p as any).name = "Eve"; }, "read-only");
raises(AttributeError, () => { (p as any).age = 1n; }, "read-only"); // also for properties that are not set
raises(AttributeError, () => { delete (p as any).name; }, "read-only");
assert(p.name === "Ada");
assert(p.schema_name() === "Person" && typeof p.identity() === "number");
{
  const alive = Array.from({ length: 100 }, () => B.Person().create());
  assert(new Set(alive.map((o) => o.identity())).size === 100); // distinct among live objects
  // Finding F12, JavaScript: identities come from a counter and are never reused, even after collection.
  const later = B.Person().create().identity();
  assert(alive.every((o) => (o.identity() as number) < (later as number)));
}
// JavaScript's own protocol probes are harmless: a proxy can be awaited, printed, and stringified.
assert(p.then === undefined && typeof p.toString === "function" && typeof String(p) === "string");
assert(!(Symbol.iterator in p) && "toString" in p && !("then" in p) && "accept" in p);

## PRX-03 · Setters: a value, or a `Spec` given a `Visitors.OfNative`

In [ ]:
const q = B.Person()
  .name("Bo")
  .age((v: any) => v.set(41n))
  .score((v: any) => v.set(1.5).clear().set(2.5)) // the Spec may chain
  .ok(false)
  .blob(new Uint8Array())
  .create();
assert(q.name === "Bo" && q.age === 41n && q.score === 2.5 && q.ok === false && q.blob.length === 0);
{
  const seen: unknown[] = [];
  B.Person(q).age((v: any) => seen.push([v.has(), v.get()]));
  assert(same((seen[0] as unknown[]), [true, 41n]));
}
B.Person(q).name((v: any) => v.clear()).update();
raises(AttributeError, () => q.name);
B.Person(q).name((v: any) => v.clear()).update(); // clearing an absent property is harmless
raises(AttributeError, () => B.Person().nickname("x"), "not a property or adjacency");
raises(AttributeError, () => B.Person()._private("x")); // names starting with "_" are never DSL setters

## PRX-04 · Finding F2 (pinned): names that collide with builder or instance members
As in Python, builder methods (`create`) and instance methods (`accept`) win over schema names, and the visitor
protocol reaches every name. Unlike Python, internals are private, so `_values` is not shadowed. JavaScript's own
members (`constructor`, `toString`) behave as schema names when declared.

In [ ]:
{
  const Odd = new S.OfObject.Builder().properties(text("create"), text("accept"), text("_values"), text("$ref"), text("toString")).create();
  Proxies.register("Odd", Odd);
  raises(TypeError, () => B.Odd().create("x")); // the builder method wins; the DSL cannot set 'create'
  const b = B.Odd();
  for (const [name, value] of [["create", "C"], ["accept", "A"], ["_values", "V"], ["$ref", "R"], ["toString", "T"]]) {
    set_property(b, name as string, value); // the visitor protocol reaches every name
  }
  const odd = b.create();
  assert(odd["$ref"] === "R");
  assert(typeof odd.accept === "function"); // the instance method wins over the property
  assert(odd._values === "V"); // no internal attribute shadows it (Python's instance does)
  assert(odd.toString === "T"); // a declared name wins over JavaScript's own member
  assert(B.Odd().toString("t").create().toString === "t"); // and is a DSL setter
  const written = (Plain.ToPlain(Odd, odd).get("objects") as Map<string, any>).get("s0");
  assert(same([...written].map(([k, v]: [string, unknown]) => `${k}=${v}`), ["create=C", "accept=A", "_values=V", "$ref=R", "toString=T"]));
}

## PRX-05 · Finalization: create / clone / update

In [ ]:
const src = B.Person().name("Src").age(1n).create();
raises(ValueError, () => B.Person(src).create());
raises(ValueError, () => B.Person().clone());
raises(ValueError, () => B.Person().update());
raises(TypeError, () => B.Person(B.Pet().name("x").create()), "not a 'Person'");
{
  const c = B.Person(src).age(2n).clone();
  assert(c !== src && c.name === "Src" && c.age === 2n && src.age === 1n);
  const u = B.Person(src).age(3n).update();
  assert(u === src && src.age === 3n);
  const reused = B.Person().name("Twin");
  const t1 = reused.create();
  const t2 = reused.create();
  assert(t1 !== t2 && t1.name === "Twin" && t2.name === "Twin");
  B.Person(src).age(99n);
  assert(src.age === 3n); // nothing changes until update()
}

## PRX-06 · Finding F8 (pinned): builders do not validate native types; serialization and validation do

In [ ]:
{
  const sloppy = B.Person().age("forty").ok(1n).create();
  assert(sloppy.age === "forty" && sloppy.ok === 1n);
  raises(TypeError, () => Plain.ToPlain(Person, sloppy));
  const problems = Validators.Validate(B)(Person, sloppy);
  assert(problems.includes("Person#0.age: expected int, got str") && problems.includes("Person#0.ok: expected bool, got int"));
}

## PRX-07 · Entries: added through adjacencies, visible from every end

In [ ]:
const ann = B.Person().name("Ann").create();
const rex = B.Pet().name("Rex").create();
B.Person(ann).pets((x: any) => x.item(rex).since("2020-01-01")).update();
assert(same(entries(Person, ann, "pets").map((e) => e.get("item")), [rex]));
assert(same(entries(Pet, rex, "owners").map((e) => `${e.get("owner") === ann}:${e.get("since")}`), ["true:2020-01-01"]));
raises(TypeError, () => B.Person().pets(rex), "entry Spec");
raises(TypeError, () => B.Person().pets({ item: rex }));
raises(AttributeError, () => B.Person().pets((x: any) => x.owner(ann)), "not a link this entry can set"); // own link is implied
raises(AttributeError, () => B.Person().pets((x: any) => x.item(rex).colour("red")));
raises(ValueError, () => B.Person().name("Nobody").pets((x: any) => x.since("never")).create(), "link 'item' is not set");
raises(TypeError, () => B.Person().pets((x: any) => x.item("Rex")), "only link proxy instances");

## PRX-08 · Entries form a set: equality follows schema equality

In [ ]:
{
  const o = B.Person().name("Set").create();
  const add = (props: Record<string, unknown> = {}) =>
    B.Person(o).pets((x: any) => { x.item(rex); for (const [k, v] of Object.entries(props)) x[k](v); return x; }).update();
  const count = () => entries(Person, o, "pets").length;
  add();
  add();
  assert(count() === 1); // an identical entry is elided
  add({ since: "a" });
  add({ since: "a" });
  add({ since: "b" });
  assert(count() === 3); // absent, "a", "b"
  add({ share: 0.0 });
  add({ share: -0.0 });
  add({ share: NaN });
  add({ share: Number("NaN") });
  assert(count() === 6); // 0.0 and -0.0 differ; all NaNs are equal
  add({ rank: 1n });
  add({ flag: true });
  assert(count() === 8); // int 1 and bool true are different entries
}

## PRX-09 · Inline creation through a link needs an unambiguous schema

In [ ]:
{
  const Lonely = new S.OfRelation.Builder().links("p", "orphan").create();
  Proxies.register("Lonely", Lonely);
  const Person2 = new S.OfObject.Builder().properties(text("name")).relations((r) => r.name("lonely").of(Lonely).me("p")).create();
  Proxies.register("Person2", Person2);
  raises(TypeError, () => B.Person().pets((x: any) => x.item((y: any) => y.name("?"))), "exactly one object schema"); // Pet and Robot
  raises(TypeError, () => B.Person2().lonely((x: any) => x.orphan((y: any) => y)), "found []"); // nothing fills 'orphan'
  const kid = B.Pet().name("Kid").create();
  const owner = B.Person().name("Own").pets((x: any) => x.item(kid))
    .trio((x: any) => x.y((y: any) => y.name("Yy")).z((z: any) => z.name("Zz"))).create();
  const trio = entries(Person, owner, "trio")[0] as Map<string, any>;
  assert(trio.get("y").schema_name() === "Pet" && trio.get("z").schema_name() === "Robot");
}

## PRX-10 · Nested inline objects are created each time the outer builder is finalized

In [ ]:
{
  const outer = B.Person().name("Outer").trio((x: any) => x.y((y: any) => y.name("Y")).z((z: any) => z.name("Z")));
  const first = outer.create();
  const second = outer.create();
  assert(entries(Person, first, "trio")[0]?.get("y") !== entries(Person, second, "trio")[0]?.get("y"));
}

## PRX-11 · Removing entries, and update() writing entries back exactly

In [ ]:
{
  const r = B.Person().name("Rem").pets((x: any) => x.item(rex).since("1")).pets((x: any) => x.item(rex).since("2"))
    .knows((x: any) => x.b(ann)).create();
  const since_of = (entry: any) => (entry.has("since") ? get_property(entry, "since") : null);
  B.Person(r).adjacency("pets", (a: any) => a.entries((e: any) => (since_of(e) === "1" ? a.remove(e) : null))).update();
  assert(same(entries(Person, r, "pets").map((e) => e.get("since")), ["2"]));
  assert(same(entries(Person, r, "knows").map((e) => e.get("b")), [ann])); // other adjacencies are untouched
  assert(entries(Pet, rex, "owners").every((e) => e.get("owner") !== r || e.get("since") === "2")); // gone from the other end
  raises(AttributeError, () => B.Person(r).adjacency("nope", () => null), "not an adjacency");
}

## PRX-12 · Finding F4 (pinned): concurrent builders — the last update() wins, entries included

In [ ]:
{
  const lw = B.Person().name("LW").create();
  const b1 = B.Person(lw).pets((x: any) => x.item(rex).rank(1n));
  const b2 = B.Person(lw).pets((x: any) => x.item(rex).rank(2n));
  b1.update();
  b2.update();
  assert(same(entries(Person, lw, "pets").map((e) => e.get("rank")), [2n]));
}

## PRX-13 · clone() copies entries; Finding F5 (pinned): a self-loop's clone links back to the original

In [ ]:
{
  const orig = B.Person().name("Orig").pets((x: any) => x.item(rex)).knows((x: any) => x.b(ann)).create();
  const cl = B.Person(orig).name("Clone").clone();
  assert(same(entries(Person, cl, "pets").map((e) => e.get("item")), [rex]));
  const owners = new Set(entries(Pet, rex, "owners").map((e) => e.get("owner").name));
  assert(owners.has("Orig") && owners.has("Clone"));

  const loop = B.Person().name("Loop").create();
  B.Person(loop).knows((x: any) => x.b(loop)).update();
  assert(same(entries(Person, loop, "knows").map((e) => e.get("b")), [loop]));
  assert(same(entries(Person, loop, "known_by").map((e) => e.get("a")), [loop]));
  const loop2 = B.Person(loop).name("Loop2").clone();
  assert(same(entries(Person, loop2, "knows").map((e) => e.get("b")), [loop]));
  assert(same(entries(Person, loop2, "known_by").map((e) => e.get("a")), [loop]));
}

## PRX-14 · Self-relations: one entry, seen through both adjacencies of the same schema

In [ ]:
{
  const x = B.Person().name("X").create();
  const y = B.Person().name("Y").knows((e: any) => e.b(x)).create();
  assert(same(entries(Person, y, "knows").map((e) => e.get("b")), [x]));
  assert(same(entries(Person, x, "known_by").map((e) => e.get("a")), [y]));
  assert(entries(Person, x, "knows").length === 0 && entries(Person, y, "known_by").length === 0);
}

## PRX-15 · The builder's visitor API reads and edits pending state

In [ ]:
{
  const vb = B.Person().name("V").age(5n);
  const names: string[] = [];
  vb.properties((prop: any) => names.push(`${prop.name()}:${prop.has()}`));
  assert(same(names, ["name:true", "age:true"]));
  assert(vb.has("name") && !vb.has("score"));
  assert(get_property(vb, "age") === 5n);
  vb.clear("age");
  assert(!vb.has("age"));
  raises(AttributeError, () => vb.property("nope", () => null), "not a property");
  const adjacencies: string[] = [];
  vb.adjacencies((a: any) => adjacencies.push(`${a.name()}:${a.me()}`));
  assert(same(adjacencies, ["pets:owner", "knows:a", "known_by:b", "trio:x"]));
}

## PRX-16 · `Proxies.OfObject.Builder` builds directly; `Proxies.OfObject.Data` is the instance type

In [ ]:
import { Fake } from "./support.js";
{
  const direct = Proxies.OfObject.Builder(Person, "Person").name("Direct").create();
  assert(direct instanceof Proxies.OfObject.Data && direct.name === "Direct");
  assert(!(Proxies.OfObject.Builder(Person, "Person") instanceof Proxies.OfObject.Data));
  assert(Proxies.OfObject.Builder(Person, "Person", direct).name("Again").update() === direct && direct.name === "Again");
  raises(TypeError, () => B.Person(new Fake("Person", { name: "fake" })), "a builder's source must be a proxy instance");
}

## PRX-17 · Entry properties must be native values (entries are keyed by them)
Object properties accept anything until serialized (F8), but an entry's values form its identity in the relation's
table, so a non-native value is refused when the entry is stored.

In [ ]:
for (const [bad, name] of [[["a"], "list"], [new Map([["k", 1n]]), "dict"], [null, "NoneType"], [Buffer.from("x"), "Buffer"]] as const) {
  raises(TypeError, () => B.Person().pets((x: any) => x.item(rex).since(bad)).create(), `an entry property must be a native value, got ${name}`);
}
{ // Bytes are native, so they key entries like any other value: equal bytes are one entry. (Keys are not type-checked
  // against the schema until serialized, so the entries are counted on the builder rather than through Plain.)
  const bytes = (s: string) => new TextEncoder().encode(s);
  const k = B.Person().pets((x: any) => x.item(rex).since(bytes("a"))).pets((x: any) => x.item(rex).since(bytes("a"))).pets((x: any) => x.item(rex).since(bytes("b"))).create();
  const found: unknown[] = [];
  B.Person(k).adjacency("pets", (a: any) => a.entries((e: unknown) => found.push(e)));
  assert(found.length === 2);
}